# 01｜金融市场与链上数据导入、清洗、日期匹配与质量检查（修正版 v02）
**负责人**：邓杰良（数据来源确认、原始数据留存、清洗、日期匹配、缺失值与完整日历检查、数据字典与来源记录）  
**研究题目**：*Is Bitcoin Still the Virtual Gold? A Re-examination of Its Hedging Capabilities Using Updated Financial Market and On-Chain Data*  
**对应指南章节**：第 6 节（数据来源）与第 7 节（数据整理、日期匹配与质量检查）

### v02 修正要点（响应组长验收反馈）
1. **完整日历日期检查**：新增针对 7×24 连续序列（`BTC_Price_USD`、`BTC_Tx_Count`）的完整公历日历缺口检查，明确记录缺失日期（如 2025-11-13 至 2025-11-15），不编造或插值。
2. **区分区间日历天数与实测链上天数**：将原 `Calendar_Days_In_Interval` 拆分为 `Calendar_Days_In_Interval`（真实间隔日历天数）与 `Observed_TX_Days`（实际获取链上数据的天数），并增加 `Interval_Complete` 完整性标记。
3. **如实记录 FTSE 100 数据来源**：清洗日志中如实标注 `Yahoo Finance (^FTSE)`，不误标为 `Eikon/Datastream`。
4. **导出 >50% 链上交易变动预警明细**：单独导出 `tx_extreme_jumps_v01.csv`，供抽查 32 次大幅变动是否源于真实网络活动或数据质量问题。


In [ ]:
# === 1. 环境初始化与路径配置 ===
import os
import json
import pathlib
import numpy as np
import pandas as pd

BASE_DIR = pathlib.Path('.')
RAW_DIR = BASE_DIR / '02_Raw_Data'
CLEAN_DIR = BASE_DIR / '03_Cleaned_Data'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_START = '2016-01-01'
SAMPLE_END = '2026-10-31'

print(f'RAW_DIR:   {RAW_DIR.resolve()}')
print(f'CLEAN_DIR: {CLEAN_DIR.resolve()}')
print(f'样本目标区间: {SAMPLE_START} 至 {SAMPLE_END}')


## 2. 定义各数据源标准化读取函数


In [ ]:
def find_raw_file(raw_dir: pathlib.Path, stem: str, exts=('.json', '.csv', '.xlsx', '.xls')) -> pathlib.Path:
    for ext in exts:
        p = raw_dir / f'{stem}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'未找到 {stem} 对应的原始文件（支持扩展名: {exts}），请检查 {raw_dir}')

def load_blockchain_series(file_path: pathlib.Path, value_col_name: str) -> pd.DataFrame:
    """读取 Blockchain.com 下载的 JSON 或 CSV 原始文件。"""
    suffix = file_path.suffix.lower()
    if suffix == '.json':
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        records = None
        if isinstance(data, dict):
            if 'values' in data and isinstance(data['values'], list):
                records = data['values']
            else:
                for k, v in data.items():
                    if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict) and 'x' in v[0] and 'y' in v[0]:
                        records = v
                        break
        elif isinstance(data, list):
            records = data
        if records is None:
            raise ValueError(f'无法识别的 Blockchain.com JSON 结构: {file_path}')
        df = pd.DataFrame(records)
        ts_sample = pd.to_numeric(df['x'], errors='coerce').dropna().iloc[0]
        unit = 'ms' if ts_sample > 1e12 else 's'
        df['Date'] = pd.to_datetime(df['x'], unit=unit, utc=True).dt.strftime('%Y-%m-%d')
        df[value_col_name] = pd.to_numeric(df['y'], errors='coerce')
        df = df[['Date', value_col_name]]
    else:
        df = pd.read_csv(file_path)
        date_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
        val_col = [c for c in df.columns if c != date_col][-1]
        df['Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')
        df[value_col_name] = pd.to_numeric(df[val_col], errors='coerce')
        df = df[['Date', value_col_name]]
    return df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

def load_fred_csv(file_path: pathlib.Path, value_col_name: str) -> pd.DataFrame:
    """读取 FRED 下载的 CSV 文件（DEXUSEU / DEXUSUK），将 '.' 转为 NaN。"""
    df = pd.read_csv(file_path, na_values=['.', 'NA', 'NaN', ''])
    date_col = [c for c in df.columns if 'date' in c.lower()][0]
    val_col = [c for c in df.columns if c != date_col][0]
    df['Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')
    df[value_col_name] = pd.to_numeric(df[val_col], errors='coerce')
    return df[['Date', value_col_name]].dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

def load_ftse_file(file_path: pathlib.Path, value_col_name: str = 'FTSE100_Level') -> pd.DataFrame:
    """读取 Yahoo Finance (^FTSE) 或 Datastream 导出的 FTSE 100 Excel/CSV 文件。"""
    if file_path.suffix.lower() in ('.xlsx', '.xls'):
        df_raw = pd.read_excel(file_path)
    else:
        df_raw = pd.read_csv(file_path)
    df = df_raw.copy()
    date_candidates = [c for c in df.columns if any(k in str(c).lower() for k in ['date', 'time', '日期'])]
    date_col = date_candidates[0] if date_candidates else df.columns[0]
    val_candidates = [c for c in df.columns if c != date_col and any(k in str(c).lower() for k in ['close', 'price', 'index', 'ftse', 'pi', 'level'])]
    val_col = val_candidates[0] if val_candidates else df.columns[1]
    df['Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')
    df[value_col_name] = pd.to_numeric(df[val_col], errors='coerce')
    return df[['Date', value_col_name]].dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)


## 3. 单序列质量检查与完整日历缺口检查
- 对于 7×24 小时运行的比特币序列（`BTC_Price_USD`、`BTC_Tx_Count`），额外对照完整公历日历 `pd.date_range(SAMPLE_START, SAMPLE_END, freq='D')`，记录缺失的具体日历日期。
- 对于 `BTC_Tx_Count` 相邻有效记录变动超过 50% 的记录，单独导出明细表 `tx_extreme_jumps_v01.csv` 供人工核查，**不直接删除**。


In [ ]:
cleaning_logs = []

def inspect_and_clean_series(df: pd.DataFrame, col_name: str, source_name: str, check_full_calendar: bool = False) -> pd.DataFrame:
    df = df.copy()
    raw_rows = len(df)
    dup_count = int(df['Date'].duplicated().sum())
    if dup_count > 0:
        df = df.drop_duplicates(subset=['Date'], keep='last')
    
    df_sub = df[(df['Date'] >= SAMPLE_START) & (df['Date'] <= SAMPLE_END)].copy()
    na_count = int(df_sub[col_name].isna().sum())
    non_pos_count = int((df_sub[col_name] <= 0).sum())
    
    df_clean = df_sub.dropna(subset=[col_name])
    df_clean = df_clean[df_clean[col_name] > 0].sort_values('Date').reset_index(drop=True)
    
    # 完整日历日期缺口检查（仅用于 7x24 连续序列）
    missing_cal_dates = []
    if check_full_calendar and len(df_clean) > 0:
        full_cal = pd.date_range(SAMPLE_START, SAMPLE_END, freq='D').strftime('%Y-%m-%d')
        existing = set(df_clean['Date'])
        missing_cal_dates = [d for d in full_cal if d not in existing]
    
    # 相邻记录 >50% 波动预警（仅预警，不删除）
    pct_chg = df_clean[col_name].pct_change()
    extreme_mask = pct_chg.abs() > 0.50
    extreme_count = int(extreme_mask.sum())
    
    if col_name == 'BTC_Tx_Count' and extreme_count > 0:
        jumps_df = pd.DataFrame({
            'Prev_Date': df_clean['Date'].shift(1)[extreme_mask],
            'Prev_Value': df_clean[col_name].shift(1)[extreme_mask],
            'Cur_Date': df_clean['Date'][extreme_mask],
            'Cur_Value': df_clean[col_name][extreme_mask],
            'Pct_Change_pct': (pct_chg[extreme_mask] * 100).round(2),
        })
        jumps_df['Calendar_Gap_Days'] = (
            pd.to_datetime(jumps_df['Cur_Date']) - pd.to_datetime(jumps_df['Prev_Date'])
        ).dt.days
        jumps_df.to_csv(CLEAN_DIR / 'tx_extreme_jumps_v01.csv', index=False, encoding='utf-8-sig')
        print(f"已导出 {col_name} >50% 变动预警明细 ({len(jumps_df)} 条): {CLEAN_DIR / 'tx_extreme_jumps_v01.csv'}")
    
    log_entry = {
        'Variable': col_name,
        'Source': source_name,
        'Raw_Rows_All': raw_rows,
        'Actual_Start': df_clean['Date'].min() if len(df_clean) else None,
        'Actual_End': df_clean['Date'].max() if len(df_clean) else None,
        'Sample_Rows_Valid': len(df_clean),
        'Duplicate_Dates_Removed': dup_count,
        'Missing_NA_Dropped': na_count,
        'Non_Positive_Dropped': non_pos_count,
        'Missing_Calendar_Days_Count': len(missing_cal_dates) if check_full_calendar else 'N/A (Trading Days)',
        'Missing_Calendar_Dates': '; '.join(missing_cal_dates[:20]) if missing_cal_dates else '',
        'Extreme_Jump_gt50pct_Flagged': extreme_count,
    }
    cleaning_logs.append(log_entry)
    print(f"[{col_name}] 有效观测: {len(df_clean)} | 缺失日历天数: {log_entry['Missing_Calendar_Days_Count']} | >50%波动预警: {extreme_count}")
    if missing_cal_dates:
        print(f"   -> 缺失的日历日期: {missing_cal_dates}")
    return df_clean

btc_price_raw = load_blockchain_series(find_raw_file(RAW_DIR, 'BTC_Market_Price_Raw'), 'BTC_Price_USD')
btc_tx_raw    = load_blockchain_series(find_raw_file(RAW_DIR, 'BTC_Transaction_Count_Raw'), 'BTC_Tx_Count')
ftse_raw      = load_ftse_file(find_raw_file(RAW_DIR, 'FTSE100_Raw', ('.xlsx', '.xls', '.csv')), 'FTSE100_Level')
usdeur_raw    = load_fred_csv(find_raw_file(RAW_DIR, 'USD_EUR_Raw', ('.csv',)), 'USD_per_EUR')
usdgbp_raw    = load_fred_csv(find_raw_file(RAW_DIR, 'USD_GBP_Raw', ('.csv',)), 'USD_per_GBP')

btc_price = inspect_and_clean_series(btc_price_raw, 'BTC_Price_USD', 'Blockchain.com Market Price (USD)', check_full_calendar=True)
btc_tx    = inspect_and_clean_series(btc_tx_raw, 'BTC_Tx_Count', 'Blockchain.com Confirmed Transactions Per Day', check_full_calendar=True)
ftse      = inspect_and_clean_series(ftse_raw, 'FTSE100_Level', 'Yahoo Finance (^FTSE Close Price Index)', check_full_calendar=False)
usdeur    = inspect_and_clean_series(usdeur_raw, 'USD_per_EUR', 'FRED DEXUSEU', check_full_calendar=False)
usdgbp    = inspect_and_clean_series(usdgbp_raw, 'USD_per_GBP', 'FRED DEXUSUK', check_full_calendar=False)


## 4. 交易日历匹配与链上指标对齐（修正区间天数计算）
- `Calendar_Days_In_Interval`：相邻两个共同金融交易日之间**真实的日历相隔天数** `(cur_date - prev_date).days`。
- `Observed_TX_Days`：该区间内**实际获取到链上数据的天数** `len(sub_vals)`。
- `Interval_Complete`：`Observed_TX_Days == Calendar_Days_In_Interval`，只有为 `True` 时才说明该区间链上覆盖完整。


In [ ]:
merged_fin = (
    btc_price
    .merge(ftse, on='Date', how='inner')
    .merge(usdeur, on='Date', how='inner')
    .merge(usdgbp, on='Date', how='inner')
    .sort_values('Date')
    .reset_index(drop=True)
)

btc_tx_sorted = btc_tx.sort_values('Date').reset_index(drop=True)
common_dates = merged_fin['Date'].tolist()
common_dt = pd.to_datetime(common_dates)

tx_same_day = []
tx_interval_mean = []
tx_interval_sum = []
calendar_days = []
observed_tx_days = []

tx_date_map = dict(zip(btc_tx_sorted['Date'], btc_tx_sorted['BTC_Tx_Count']))

for i, cur_d in enumerate(common_dates):
    tx_same_day.append(tx_date_map.get(cur_d, np.nan))
    if i == 0:
        calendar_days.append(1)
        obs_cnt = 1 if cur_d in tx_date_map else 0
        observed_tx_days.append(obs_cnt)
        tx_interval_mean.append(tx_date_map.get(cur_d, np.nan))
        tx_interval_sum.append(tx_date_map.get(cur_d, np.nan))
    else:
        prev_d = common_dates[i - 1]
        cal_gap = int((common_dt[i] - common_dt[i - 1]).days)
        mask = (btc_tx_sorted['Date'] > prev_d) & (btc_tx_sorted['Date'] <= cur_d)
        sub_vals = btc_tx_sorted.loc[mask, 'BTC_Tx_Count']
        calendar_days.append(cal_gap)
        observed_tx_days.append(int(len(sub_vals)))
        tx_interval_mean.append(sub_vals.mean() if len(sub_vals) > 0 else np.nan)
        tx_interval_sum.append(sub_vals.sum() if len(sub_vals) > 0 else np.nan)

merged_fin['BTC_Tx_Count'] = tx_same_day
merged_fin['BTC_Tx_Count_Mean'] = tx_interval_mean
merged_fin['BTC_Tx_Count_Sum'] = tx_interval_sum
merged_fin['Calendar_Days_In_Interval'] = calendar_days
merged_fin['Observed_TX_Days'] = observed_tx_days
merged_fin['Interval_Complete'] = (merged_fin['Observed_TX_Days'] == merged_fin['Calendar_Days_In_Interval'])

incomplete_n = int((~merged_fin['Interval_Complete']).sum())
print(f"共同有效交易日样本量: {len(merged_fin)} 行 | 区间链上数据不完整行数: {incomplete_n}")


## 5. 对数收益率与链上变化率计算


In [ ]:
for price_col, ret_col in [
    ('BTC_Price_USD', 'r_BTC'),
    ('FTSE100_Level', 'r_FTSE'),
    ('USD_per_EUR',   'r_USDEUR'),
    ('USD_per_GBP',   'r_USDGBP'),
]:
    merged_fin[ret_col] = 100.0 * (np.log(merged_fin[price_col]) - np.log(merged_fin[price_col].shift(1)))

merged_fin['dln_BTC_Tx_Count'] = 100.0 * (np.log(merged_fin['BTC_Tx_Count']) - np.log(merged_fin['BTC_Tx_Count'].shift(1)))
merged_fin['dln_BTC_Tx_Count_Mean'] = 100.0 * (np.log(merged_fin['BTC_Tx_Count_Mean']) - np.log(merged_fin['BTC_Tx_Count_Mean'].shift(1)))


## 6. 质量验收检查与文件导出


In [ ]:
assert merged_fin['Date'].is_unique, "错误：共同日期中存在重复值！"
assert merged_fin['Date'].is_monotonic_increasing, "错误：日期未按升序排列！"

core_cols = ['BTC_Price_USD', 'FTSE100_Level', 'USD_per_EUR', 'USD_per_GBP', 'BTC_Tx_Count']
missing_core = merged_fin[core_cols].isna().sum().to_dict()
print("核心字段缺失值统计:", missing_core)

clean_csv_path = CLEAN_DIR / 'merged_clean_v01.csv'
log_csv_path = CLEAN_DIR / 'cleaning_log_v01.csv'

merged_fin.to_csv(clean_csv_path, index=False, encoding='utf-8-sig')
pd.DataFrame(cleaning_logs).to_csv(log_csv_path, index=False, encoding='utf-8-sig')

print(f"已导出清洗数据: {clean_csv_path} (共 {len(merged_fin)} 行, {len(merged_fin.columns)} 列)")
print(f"已导出清洗日志: {log_csv_path}")
